# 1: Imports and Setup

In [6]:
!pip install -U ultralytics

In [7]:
import os
import yaml
import torch
import gdown
import zipfile
from pathlib import Path
from ultralytics import YOLO

# 1. Path Configuration (Portable Root)
# Path.cwd() automatically finds the folder where you run the script
base_path = Path.cwd()
dataset_path = base_path / "vehicle_dataset"
yaml_path = dataset_path / "data.yaml"
zip_destination = base_path / "vehicle_dataset.zip"

# Google Drive File ID from your link
GDRIVE_FILE_ID = "10KkvyF1zh6NjJNP1abdxTYThwlRjwHsg"

# 2. Check and Download Dataset
if not dataset_path.exists():
    print(f"Dataset not found at {dataset_path}. Downloading...")
    url = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"

    # Download using gdown
    gdown.download(url, str(zip_destination), quiet=False)

    print("Extracting dataset...")
    with zipfile.ZipFile(zip_destination, "r") as zip_ref:
        zip_ref.extractall(base_path)

    # Clean up zip file
    if zip_destination.exists():
        os.remove(zip_destination)
    print("Dataset ready.")
else:
    print(f"Dataset already exists at {dataset_path}. Skipping download.")

# 3. Check Hardware
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if cuda_available:
    print(f"Using Device: {torch.cuda.get_device_name(0)}")

# 4. Smart Path Detection for data.yaml
# We check if 'train/images' exists; if not, we fallback to just 'images'
# to prevent the FileNotFoundError you encountered.
train_rel_path = (
    "train/images" if (dataset_path / "train/images").exists() else "images"
)
val_rel_path = "valid/images" if (dataset_path / "valid/images").exists() else "images"

dataset_config = {
    "path": str(dataset_path.absolute()),
    "train": train_rel_path,
    "val": val_rel_path,
    "nc": 1,  # Change this to 1
    "names": ["vehicle"],  # Only include the class you actually have
}

# Ensure directory exists before writing YAML
dataset_path.mkdir(parents=True, exist_ok=True)
with open(yaml_path, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"--- Setup Complete ---")
print(f"Root: {base_path}")
print(f"YAML: {yaml_path}")
print(f"Train path set to: {train_rel_path}")

Dataset already exists at /content/vehicle_dataset. Skipping download.
CUDA Available: False
--- Setup Complete ---
Root: /content
YAML: /content/vehicle_dataset/data.yaml
Train path set to: images


# 2: Load and Train YOLOv8s Model

In [8]:
# from ultralytics import YOLO

model = YOLO("yolov8s")   # auto-download clean model

results = model.train(
    data="vehicle_dataset/data.yaml",
    epochs=100,
    imgsz=640,

    batch=24,
    device="mps",

    workers=2,
    cache="ram",
    amp=True,

    patience=10,   # early stopping

    optimizer="auto",
    pretrained=True,

    project="runs/train",   # better for local (NOT /content)
    name="vehicle_exp",
    exist_ok=True,

    save=True,
    save_period=5,
    plots=True,
    verbose=True
)

Ultralytics 8.4.34 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=vehicle_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicle_exp, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspectiv

: 

: 

: 

# 3: Validate the Trained Model

In [ ]:
val_results = model.val(data="vehicle_dataset/data.yaml", split="test")
print("Validation results:")
print(val_results)

# 4: Predict on Sample Images (Optional)

In [ ]:
from ultralytics import YOLO
import os
import random
import matplotlib.pyplot as plt
import cv2

# Load your trained model
model = YOLO("models/yolov5_1k.pt")  # adjust path

# Path to test images folder
test_folder = "vehicle_dataset/images/test/"

# Get all image files in the folder
all_images = [
    os.path.join(test_folder, f)
    for f in os.listdir(test_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# Pick 10 random images
test_images = random.sample(all_images, min(10, len(all_images)))

# Run inference and display
for img_path in test_images:
    results = model.predict(
        source=img_path, conf=0.5, save=False
    )  # save=False, we just display
    annotated_img = results[0].plot()  # draw boxes
    annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 6))
    plt.imshow(annotated_img)
    plt.title(f"Predictions for {os.path.basename(img_path)}")
    plt.axis("off")
    plt.show()

    # Print box info
    for result in results:
        print("Boxes:", result.boxes.xyxy)
        print("Confidence:", result.boxes.conf)
        print("Class:", result.boxes.cls)